In [1]:

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


In [4]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')


In [5]:
# Connect to client libraries

openai = OpenAI()



In [ ]:
supported_languages = ["typescript", "javascript", "php"]

In [ ]:
system_prompt = f""" Student with background in typescript, javascript and php are learning AI engineering with python, they need to be able to easily understand the python code in the language they understand. Your job is to be the expert code converter who converts the python code to the specified language by follwoing these rules: 
- Produce clean, idiomatic code in the target language
- Preserve the same logic and output
- Add type annotations where the language supports them
- Respond ONLY with the converted code, no explanations
 """

In [ ]:
def convert_code(target_language, python_code):
    if not python_code.strip():
        yield "Please paste some Python code."
        return
    
    prompt = f"Convert the following Python code to {target_language}:\n{python_code}"

    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":prompt}
        ],
        stream=True,
        temperature=0.2,
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

    

In [ ]:
SAMPLE_PYTHON = """def convert_code(target_language, python_code):
    if not python_code.strip():
        yield "Please paste some Python code."
        return
    
    prompt = f"Convert the following Python code to {target_language}:\n{python_code}"

    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content":prompt}
        ],
        stream=True,
        temperature=0.2,
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
"""

gr.Interface(
    fn=convert_code,
    inputs=[
        gr.Code(label="Python Code", language="python", value=SAMPLE_PYTHON, lines=15),
        gr.Dropdown(["TypeScript", "JavaScript", "PHP"], label="Convert to", value="TypeScript")
    ],
    outputs=gr.Markdown(label="Converted Code"),
    title="Python Code Converter",
    description="Convert Python to TypeScript, JavaScript, or PHP",
    flagging_mode="never"
).launch()